# NB2: Paralelismo en CPU

**Computación de Altas Prestaciones para Ciencia de Datos (CAPCD)**

---

## Objetivos de este notebook

1. Entender la diferencia entre concurrencia y paralelismo.
2. Comprender el **GIL** de Python y sus implicaciones.
3. Usar `ThreadPoolExecutor` para tareas *I/O bound*.
4. Usar `ProcessPoolExecutor` para tareas *CPU bound*.
5. Saber cuándo usar cada uno.

---

## 1. El GIL (Global Interpreter Lock)

Python tiene un mecanismo llamado **GIL** que permite que solo un hilo ejecute código Python a la vez.

### ¿Por qué existe?
Para proteger las estructuras internas de CPython (el intérprete más común) de condiciones de carrera.

### Consecuencia
- **Tareas CPU-bound** (cálculos matemáticos): los hilos (`threading`) **NO** dan speedup real.
- **Tareas I/O-bound** (red, disco, APIs): los hilos **SÍ** funcionan porque el GIL se libera durante la espera.

### Solución para CPU-bound
Usar **procesos** en lugar de hilos. Cada proceso tiene su propio intérprete Python y su propio GIL.

| Tipo de tarea | Ejemplo | Herramienta |
|---|---|---|
| **I/O bound** | Descargar datos, leer archivos | `ThreadPoolExecutor` |
| **CPU bound** | Cálculos numéricos, procesamiento | `ProcessPoolExecutor` |

---

## 2. `concurrent.futures`: una API unificada

El módulo `concurrent.futures` de la librería estándar ofrece una interfaz idéntica para hilos y procesos:

```python
from concurrent.futures import ThreadPoolExecutor  # hilos
from concurrent.futures import ProcessPoolExecutor  # procesos

with ThreadPoolExecutor(max_workers=4) as executor:
    resultados = executor.map(funcion, datos)
```

Cambiar de hilos a procesos es cambiar una palabra: `ThreadPoolExecutor` → `ProcessPoolExecutor`.

### Métodos principales

| Método | Uso |
|---|---|
| `executor.map(fn, iterables)` | Aplica `fn` a cada elemento (como `map()` pero en paralelo) |
| `executor.submit(fn, *args)` | Envía una tarea individual, devuelve un `Future` |
| `future.result()` | Obtiene el resultado de un `Future` (bloqueante) |

---

## Demo 1: Debido al GIL, los hilos NO aceleran un trabajo *CPU bound*

In [ ]:
import time
import math
import multiprocessing
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor


def trabajo_cpu(n):
    """Tarea CPU-bound: cálculo matemático intensivo."""
    total = 0.0
    for i in range(n):
        total += math.sqrt(i) * math.sin(i)
    return total


N = 50_000_000
CHUNKS = 2  # Dividimos el trabajo en 2 partes
tareas = [N // CHUNKS] * CHUNKS

# --- Secuencial ---
start = time.perf_counter()
resultados_seq = [trabajo_cpu(n) for n in tareas]
t_seq = time.perf_counter() - start
print(f"Secuencial:        {t_seq:.3f}s")

# --- Hilos (ThreadPoolExecutor) ---
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=2) as executor:
    resultados_threads = list(executor.map(trabajo_cpu, tareas))
t_threads = time.perf_counter() - start
print(f"Hilos (2 threads):  {t_threads:.3f}s  (speedup: {t_seq/t_threads:.2f}x)")

# --- Procesos (ProcessPoolExecutor) ---
# Usamos 'fork' explícitamente: en Python 3.12+ el método por defecto en Linux
# cambió a 'forkserver', que no funciona bien con funciones definidas en Jupyter.
mp_ctx = multiprocessing.get_context("fork")
start = time.perf_counter()
with ProcessPoolExecutor(max_workers=2, mp_context=mp_ctx) as executor:
    resultados_procs = list(executor.map(trabajo_cpu, tareas))
t_procs = time.perf_counter() - start
print(f"Procesos (2 procs): {t_procs:.3f}s  (speedup: {t_seq/t_procs:.2f}x)")


### Resultado esperado

- **Hilos**: *speedup* ≈ 1x (el GIL bloquea la ejecución paralela real)
- **Procesos**: *speedup* ≈ 2x (cada proceso tiene su propio GIL)

En entornos con pocos núcleos como Google Colab, el *speedup* máximo con procesos estará acotado por el número de núcleos físicos disponibles. Tened en cuenta que las máquinas virtuales de Colab sufren de mucha contención, por lo que puede que obtengáis incluso una penalización de rendimiento (*speedup* < 1x).


---

## Demo 2: Los hilos SÍ aceleran un trabajo *I/O bound*

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor


def tarea_io(segundos):
    """Simula una tarea I/O bound (descarga, lectura de disco, API call)."""
    time.sleep(segundos)  # Simula espera por I/O
    return f"Tarea completada en {segundos}s"


tareas_io = [0.5] * 8  # 8 tareas de 0.5 segundos cada una

# --- Secuencial ---
start = time.perf_counter()
for t in tareas_io:
    tarea_io(t)
t_seq = time.perf_counter() - start
print(f"Secuencial: {t_seq:.2f}s")

# --- Paralelo con hilos ---
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as executor:
    resultados = list(executor.map(tarea_io, tareas_io))
t_par = time.perf_counter() - start
print(f"Hilos (8):  {t_par:.2f}s  (speedup: {t_seq/t_par:.1f}x)")

8 tareas de 0.5s en secuencial = 4s. Con 8 hilos = ~0.5s. Speedup ≈ 8x.

---

## Demo 3: `executor.submit()` y `Future`

Cuando las tareas tienen distintos argumentos o queremos más control, usamos `submit()` en lugar de `map()`.

In [ ]:
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import time


def procesar_bloque(bloque_id, datos):
    """Procesa un bloque de datos: calcula media y desviación."""
    media = np.mean(datos)
    std = np.std(datos)
    return {"bloque": bloque_id, "media": media, "std": std}


# Crear datos divididos en bloques
datos_completos = np.random.rand(4_000_000)
bloques = np.array_split(datos_completos, 4)

mp_ctx = multiprocessing.get_context("fork")
start = time.perf_counter()
with ProcessPoolExecutor(max_workers=4, mp_context=mp_ctx) as executor:
    # Enviar tareas con submit()
    futures = {
        executor.submit(procesar_bloque, i, bloque): i
        for i, bloque in enumerate(bloques)
    }

    # Recoger resultados a medida que terminan
    for future in as_completed(futures):
        resultado = future.result()
        print(f"  Bloque {resultado['bloque']}: "
              f"media={resultado['media']:.4f}, std={resultado['std']:.4f}")

t_total = time.perf_counter() - start
print(f"\nTiempo total: {t_total:.3f}s")


### `map()` vs `submit()`

| | `map()` | `submit()` |
|---|---|---|
| Uso | Misma función, distintos datos | Control individual por tarea |
| Orden | Mantiene el orden de entrada | `as_completed()` devuelve por orden de finalización |
| Sintaxis | Más simple | Más flexible |

---

## 3. ¿Cómo saber qué usar?

```
¿Tu tarea es lenta por...?
│
├── Esperar I/O (red, disco, API)   →  ThreadPoolExecutor
│
├── Cálculos matemáticos            →  ProcessPoolExecutor
│    (si son NumPy puros, ya liberan el GIL → los threads pueden funcionar)
│
└── Bucles numéricos en Python puro →  Mejor usar Numba (NB3) o GPU (NB4-NB5)
```

### Limitaciones de `ProcessPoolExecutor`

- **Coste de serialización**: Los datos se copian entre procesos (pickle). Arrays grandes = lento.
- **Overhead de creación**: Crear procesos es más caro que crear hilos.
- **Memoria**: Cada proceso duplica la memoria del programa.

---

## Ejercicio 1: Paralelizar un cálculo CPU-bound

La función `es_primo()` comprueba si un número es primo. Úsala para encontrar los primos en una lista de números grandes, primero en secuencial y luego con `ProcessPoolExecutor`.

In [ ]:
import math
import time
import multiprocessing
from concurrent.futures import ProcessPoolExecutor


def es_primo(n):
    """Comprueba si n es primo (intencionalmente sin optimizar)."""
    if n < 2:
        return False
    if n < 4:
        return True
    if n % 2 == 0 or n % 3 == 0:
        return False
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0:
            return False
        i += 6
    return True


# Números grandes para comprobar
numeros = [
    112272535095293, 112582705942171, 112272535095293,
    115280095190773, 115797848077099, 1099726899285419,
    112272535095293, 112582705942171, 112272535095293,
    115280095190773, 115797848077099, 1099726899285419,
]

# --- Secuencial ---
start = time.perf_counter()
resultados_seq = [es_primo(n) for n in numeros]
t_seq = time.perf_counter() - start
print(f"Secuencial: {t_seq:.3f}s")

# --- TODO: Paralelo con ProcessPoolExecutor ---
mp_ctx = multiprocessing.get_context("fork")
start = time.perf_counter()
# TODO: Usa ProcessPoolExecutor (con mp_context=mp_ctx) y executor.map() para paralelizar
resultados_par = []  # TODO: reemplaza con el resultado real
t_par = time.perf_counter() - start
if resultados_par:
    print(f"Paralelo:   {t_par:.3f}s  (speedup: {t_seq/t_par:.2f}x)")
else:
    print(f"Paralelo:   pendiente (implementa el TODO)")


In [ ]:
# Autoevaluación
assert resultados_par == resultados_seq, "Los resultados paralelos deben coincidir con los secuenciales"
assert t_par < t_seq, "La versión paralela debería ser más rápida"
print("✅ ¡Correcto! Los resultados coinciden y la versión paralela es más rápida.")

---

## Ejercicio 2: Descargas paralelas con hilos

Simula la descarga de 10 URLs (con `time.sleep`) y compara secuencial vs paralelo con `ThreadPoolExecutor`.

In [ ]:
import time
import random
from concurrent.futures import ThreadPoolExecutor


def descargar_url(url):
    """Simula la descarga de una URL (I/O bound)."""
    tiempo_descarga = random.uniform(0.2, 0.8)  # Simula latencia variable
    time.sleep(tiempo_descarga)
    return f"{url}: {tiempo_descarga:.2f}s"


urls = [f"https://api.example.com/data/{i}" for i in range(10)]

# --- Secuencial ---
start = time.perf_counter()
resultados_seq = [descargar_url(url) for url in urls]
t_seq = time.perf_counter() - start
print(f"Secuencial: {t_seq:.2f}s")

# --- TODO: Paralelo con ThreadPoolExecutor ---
start = time.perf_counter()
# TODO: Usa ThreadPoolExecutor con max_workers=5 y executor.map()
resultados_par = []  # TODO: reemplaza
t_par = time.perf_counter() - start
if resultados_par:
    print(f"Hilos (5):  {t_par:.2f}s  (speedup: {t_seq/t_par:.1f}x)")
else:
    print(f"Hilos (5):  pendiente (implementa el TODO)")

In [ ]:
# Autoevaluación
assert len(resultados_par) == 10, "Deben haber 10 resultados"
assert t_par < t_seq * 0.6, "El speedup debería ser significativo (>1.5x)"
print("✅ ¡Correcto! Las descargas paralelas son mucho más rápidas.")

---

## Ejercicio 3: Procesamiento por bloques con `submit()`

Tienes un array grande de datos. Divídelo en bloques, procesa cada bloque en paralelo con `submit()`, y recoge los resultados con `as_completed()`.

In [ ]:
import numpy as np
import time
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed


def calcular_histograma(datos, bins=50):
    """Calcula un histograma parcial de los datos."""
    hist, _ = np.histogram(datos, bins=bins, range=(0, 1))
    return hist


# Datos grandes
datos = np.random.rand(10_000_000)
n_bloques = 4

# --- Secuencial ---
start = time.perf_counter()
histograma_total = calcular_histograma(datos)
t_seq = time.perf_counter() - start
print(f"Secuencial: {t_seq:.3f}s")

# --- TODO: Paralelo ---
# 1. Divide 'datos' en n_bloques partes con np.array_split()
# 2. Usa ProcessPoolExecutor (con mp_context=mp_ctx) y submit() para calcular histograma de cada bloque
# 3. Suma los histogramas parciales para obtener el histograma total

mp_ctx = multiprocessing.get_context("fork")
start = time.perf_counter()
# TODO: Implementa aquí
histograma_paralelo = None  # TODO: reemplaza
t_par = time.perf_counter() - start
print(f"Paralelo:   {t_par:.3f}s")


In [ ]:
# Autoevaluación
assert histograma_paralelo is not None, "Debes calcular el histograma paralelo"
assert np.array_equal(histograma_paralelo, histograma_total), "Los histogramas deben coincidir"
print("✅ ¡Correcto! El histograma paralelo coincide con el secuencial.")

---

## El futuro: Python sin GIL

El GIL es la mayor limitación del paralelismo en Python, pero está cambiando:

- **PEP 703** (aceptado 2023): propone un build de CPython sin GIL (*free-threaded Python*).
- **Python 3.13** (2024): primera versión con builds experimentales sin GIL, instalables como `python3.13t` y activables en tiempo de ejecución con `-X gil=0`.
- **~2028**: se espera que el modo sin GIL sea la opción por defecto.

Cuando esto llegue, `ThreadPoolExecutor` podrá acelerar **tareas CPU-bound** de verdad, no solo I/O-bound. Esto simplificará enormemente el código paralelo: no haría falta `ProcessPoolExecutor` (con su overhead de pickle y memoria duplicada).

> **¿Por qué no se quita ya?** Librerías como NumPy, Pandas y scikit-learn tienen código C compilado que asume el GIL. Migrar todo el ecosistema es un proceso gradual para evitar romper nada (nadie quiere otra transición Python 2→3).

Por ahora, las estrategias que hemos visto en este notebook (`ThreadPoolExecutor` para I/O, `ProcessPoolExecutor` para CPU) siguen siendo la forma correcta de paralelizar.

---

## Resumen

| Concepto | Detalle |
|---|---|
| **GIL** | Solo un hilo ejecuta Python a la vez → hilos no aceleran CPU-bound |
| **ThreadPoolExecutor** | Para tareas I/O-bound (red, disco) |
| **ProcessPoolExecutor** | Para tareas CPU-bound (cálculos) |
| **`executor.map()`** | Paralelo + simple (misma función, distintos datos) |
| **`executor.submit()`** | Paralelo + flexible (control individual) |

### Limitaciones

- `ProcessPoolExecutor` tiene overhead por serialización (pickle) y creación de procesos.
- Para bucles numéricos puros, **Numba** (siguiente notebook) es mucho más eficiente.

### Siguiente paso
En el **NB3** veremos cómo Numba puede compilar bucles Python a código máquina y paralelizarlos sin el overhead de procesos.